In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# TVP-VAR / GFEVD Diagnostic Check
# ------------------------------------------------------------
# 목적
# 1) beta_t 방향 문제가 있는지 점검
# 2) cov_t가 innovation covariance처럼 보이는지 점검
# 3) A_t 안정성(roots / spectral radius) 점검
# 4) GFEVD 결과가 diagonal dominance를 가지는지 점검
# 5) beta 방향(normal vs transpose)에 따라 GFEVD/TCI/NET이 어떻게 달라지는지 비교
#
# 입력 파일
#   - ./tvpvar_beta.npy
#   - ./tvpvar_cov.npy
#   - ./gfevd_all.npy   (있으면 기존 결과 진단)
#   - ./tvpvar_selected_lag.txt
#
# 출력 파일
#   - ./diagnostic_output/diagnostic_summary.txt
#   - ./diagnostic_output/beta_orientation_compare.csv
#   - ./diagnostic_output/covariance_summary.csv
#   - ./diagnostic_output/stability_summary.csv
#   - ./diagnostic_output/gfevd_existing_summary.csv
# ============================================================

# =========================
# Config
# =========================
BASE_DIR = Path("./")
OUT_DIR = BASE_DIR / "diagnostic_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BETA_FILE = BASE_DIR / "tvpvar_beta.npy"
COV_FILE = BASE_DIR / "tvpvar_cov.npy"
GFEVD_FILE = BASE_DIR / "gfevd_all.npy"
LAG_FILE = BASE_DIR / "tvpvar_selected_lag.txt"

OUT_SUMMARY = OUT_DIR / "diagnostic_summary.txt"
OUT_ORIENT = OUT_DIR / "beta_orientation_compare.csv"
OUT_COV = OUT_DIR / "covariance_summary.csv"
OUT_STAB = OUT_DIR / "stability_summary.csv"
OUT_GFEVD = OUT_DIR / "gfevd_existing_summary.csv"

VAR_NAMES = [
    "SOLVPN",
    "COPPER",
    "GOLD",
    "SILVER",
    "DXY",
    "UST10Y",
    "VIX"
]

H = 10
MAX_CHECK_T = 30   # 마지막 몇 개 시점만 orientation 비교할지
EPS = 1e-12

# =========================
# Helper
# =========================
def read_lag_default_1(path: Path) -> int:
    if path.exists():
        txt = path.read_text(encoding="utf-8").strip()
        try:
            return int(txt)
        except Exception:
            pass
    return 1


def row_normalize(mat: np.ndarray) -> np.ndarray:
    rs = mat.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    return mat / rs


def unpack_A_mats(beta_one_t: np.ndarray, p: int, use_transpose: bool = False):
    """
    beta_one_t shape assumed: (k, k*p)
    """
    k, kp = beta_one_t.shape
    if kp != k * p:
        raise ValueError(f"beta_one_t.shape={beta_one_t.shape}, expected second dim = k*p = {k*p}")

    A_list = []
    for lag in range(p):
        start = lag * k
        end = (lag + 1) * k
        block = beta_one_t[:, start:end]
        if use_transpose:
            block = block.T
        A_list.append(block)
    return A_list


def compute_phi_list(A_list, H: int):
    """
    VAR(p) -> VMA(Phi_h), Phi_0 ... Phi_{H-1}
    """
    k = A_list[0].shape[0]
    p = len(A_list)

    phi = [np.eye(k)]
    for h in range(1, H):
        phi_h = np.zeros((k, k))
        for j in range(1, min(h, p) + 1):
            phi_h += A_list[j - 1] @ phi[h - j]
        phi.append(phi_h)
    return phi


def compute_gfevd(A_list, Sigma_t: np.ndarray, H: int):
    """
    Generalized FEVD
    row = response i
    col = shock source j
    """
    k = Sigma_t.shape[0]
    phi = compute_phi_list(A_list, H)

    gfevd = np.zeros((k, k), dtype=float)

    for i in range(k):
        for j in range(k):
            sigma_jj = Sigma_t[j, j]
            if sigma_jj <= EPS:
                continue

            numer = 0.0
            denom = 0.0

            e_i = np.zeros((k, 1))
            e_i[i, 0] = 1.0

            e_j = np.zeros((k, 1))
            e_j[j, 0] = 1.0

            for h in range(H):
                phi_h = phi[h]

                term_ij = e_i.T @ phi_h @ Sigma_t @ e_j
                numer += float(term_ij) ** 2

                term_ii = e_i.T @ phi_h @ Sigma_t @ phi_h.T @ e_i
                denom += float(term_ii)

            if denom > EPS:
                gfevd[i, j] = (numer / sigma_jj) / denom

    return row_normalize(gfevd)


def compute_tci_net(theta: np.ndarray):
    diag = np.diag(theta)
    from_ = theta.sum(axis=1) - diag
    to_ = theta.sum(axis=0) - diag
    net_ = to_ - from_
    tci_ = (theta.sum() - np.trace(theta)) / theta.shape[0] * 100.0
    return tci_, to_, from_, net_


def companion_matrix(A_list):
    """
    A_list: [A1, A2, ..., Ap], each k x k
    return companion matrix (k*p, k*p)
    """
    k = A_list[0].shape[0]
    p = len(A_list)

    if p == 1:
        return A_list[0].copy()

    top = np.hstack(A_list)
    bottom = np.hstack([
        np.eye(k * (p - 1)),
        np.zeros((k * (p - 1), k))
    ])
    return np.vstack([top, bottom])


def spectral_radius(A_list):
    comp = companion_matrix(A_list)
    vals = np.linalg.eigvals(comp)
    return float(np.max(np.abs(vals)))


def summarize_matrix(theta: np.ndarray):
    k = theta.shape[0]
    diag_mean = float(np.mean(np.diag(theta)))
    offdiag_mean = float((theta.sum() - np.trace(theta)) / (k * (k - 1)))
    row_err = float(np.max(np.abs(theta.sum(axis=1) - 1.0)))
    return diag_mean, offdiag_mean, row_err


# =========================
# Load
# =========================
if not BETA_FILE.exists():
    raise FileNotFoundError(f"Not found: {BETA_FILE}")
if not COV_FILE.exists():
    raise FileNotFoundError(f"Not found: {COV_FILE}")

beta_t = np.load(BETA_FILE)
cov_t = np.load(COV_FILE)
p = read_lag_default_1(LAG_FILE)

if beta_t.ndim != 3:
    raise ValueError(f"tvpvar_beta.npy must be 3D, got {beta_t.shape}")
if cov_t.ndim != 3:
    raise ValueError(f"tvpvar_cov.npy must be 3D, got {cov_t.shape}")

T_beta, k1, kp = beta_t.shape
T_cov, k2, k3 = cov_t.shape

if T_beta != T_cov:
    raise ValueError(f"beta_t and cov_t time dim mismatch: {T_beta} vs {T_cov}")
if k2 != k3:
    raise ValueError(f"cov_t must be square per time, got {cov_t.shape}")
if k1 != k2:
    raise ValueError(f"beta_t first dim and cov_t size mismatch: {beta_t.shape}, {cov_t.shape}")

k = k1

print(f"[INFO] beta_t shape = {beta_t.shape}")
print(f"[INFO] cov_t  shape = {cov_t.shape}")
print(f"[INFO] lag p       = {p}")

if len(VAR_NAMES) != k:
    print(f"[WARN] len(VAR_NAMES)={len(VAR_NAMES)} but k={k}. Variable names will not be used for shape checks.")

# =========================
# 1) cov_t 진단
# =========================
cov_rows = []
for t in range(T_cov):
    S = cov_t[t]
    diag_vals = np.diag(S)
    eigvals = np.linalg.eigvalsh((S + S.T) / 2.0)

    cov_rows.append({
        "t": t,
        "diag_min": float(np.min(diag_vals)),
        "diag_max": float(np.max(diag_vals)),
        "diag_mean": float(np.mean(diag_vals)),
        "offdiag_abs_mean": float(np.mean(np.abs(S - np.diag(np.diag(S))))),
        "symmetry_max_abs_diff": float(np.max(np.abs(S - S.T))),
        "eig_min": float(np.min(eigvals)),
        "eig_max": float(np.max(eigvals)),
        "is_psd_like": bool(np.min(eigvals) >= -1e-8),
    })

df_cov = pd.DataFrame(cov_rows)
df_cov.to_csv(OUT_COV, index=False)

# =========================
# 2) beta orientation 비교
# =========================
check_idx = list(range(max(0, T_beta - MAX_CHECK_T), T_beta))

orient_rows = []
stab_rows = []

for t in check_idx:
    B = beta_t[t]
    S = cov_t[t]

    # normal
    try:
        A_norm = unpack_A_mats(B, p, use_transpose=False)
        rho_norm = spectral_radius(A_norm)
        g_norm = compute_gfevd(A_norm, S, H)
        tci_norm, to_norm, from_norm, net_norm = compute_tci_net(g_norm)
        diag_norm, offdiag_norm, rowerr_norm = summarize_matrix(g_norm)
    except Exception as e:
        rho_norm = np.nan
        tci_norm = np.nan
        diag_norm = np.nan
        offdiag_norm = np.nan
        rowerr_norm = np.nan
        net_norm = np.full(k, np.nan)
        err_norm = str(e)
    else:
        err_norm = ""

    # transpose
    try:
        A_tr = unpack_A_mats(B, p, use_transpose=True)
        rho_tr = spectral_radius(A_tr)
        g_tr = compute_gfevd(A_tr, S, H)
        tci_tr, to_tr, from_tr, net_tr = compute_tci_net(g_tr)
        diag_tr, offdiag_tr, rowerr_tr = summarize_matrix(g_tr)
    except Exception as e:
        rho_tr = np.nan
        tci_tr = np.nan
        diag_tr = np.nan
        offdiag_tr = np.nan
        rowerr_tr = np.nan
        net_tr = np.full(k, np.nan)
        err_tr = str(e)
    else:
        err_tr = ""

    orient_rows.append({
        "t": t,
        "TCI_normal": tci_norm,
        "TCI_transpose": tci_tr,
        "diag_mean_normal": diag_norm,
        "diag_mean_transpose": diag_tr,
        "offdiag_mean_normal": offdiag_norm,
        "offdiag_mean_transpose": offdiag_tr,
        "diag_minus_offdiag_normal": (diag_norm - offdiag_norm) if pd.notna(diag_norm) else np.nan,
        "diag_minus_offdiag_transpose": (diag_tr - offdiag_tr) if pd.notna(diag_tr) else np.nan,
        "row_err_normal": rowerr_norm,
        "row_err_transpose": rowerr_tr,
        "net_abs_mean_normal": float(np.nanmean(np.abs(net_norm))),
        "net_abs_mean_transpose": float(np.nanmean(np.abs(net_tr))),
        "rho_normal": rho_norm,
        "rho_transpose": rho_tr,
        "stable_normal": bool(pd.notna(rho_norm) and rho_norm < 1.0),
        "stable_transpose": bool(pd.notna(rho_tr) and rho_tr < 1.0),
        "error_normal": err_norm,
        "error_transpose": err_tr,
    })

    stab_rows.append({
        "t": t,
        "rho_normal": rho_norm,
        "rho_transpose": rho_tr,
        "stable_normal": bool(pd.notna(rho_norm) and rho_norm < 1.0),
        "stable_transpose": bool(pd.notna(rho_tr) and rho_tr < 1.0),
    })

df_orient = pd.DataFrame(orient_rows)
df_stab = pd.DataFrame(stab_rows)

df_orient.to_csv(OUT_ORIENT, index=False)
df_stab.to_csv(OUT_STAB, index=False)

# =========================
# 3) 기존 gfevd_all 진단
# =========================
gfevd_summary_lines = []
if GFEVD_FILE.exists():
    gfevd_all = np.load(GFEVD_FILE)
    if gfevd_all.ndim != 3:
        raise ValueError(f"gfevd_all.npy must be 3D, got {gfevd_all.shape}")

    g_rows = []
    for t in range(gfevd_all.shape[0]):
        theta = gfevd_all[t]
        tci_, to_, from_, net_ = compute_tci_net(theta)
        diag_mean, offdiag_mean, row_err = summarize_matrix(theta)

        g_rows.append({
            "t": t,
            "TCI": tci_,
            "diag_mean": diag_mean,
            "offdiag_mean": offdiag_mean,
            "diag_minus_offdiag": diag_mean - offdiag_mean,
            "row_err": row_err,
            "net_abs_mean": float(np.mean(np.abs(net_))),
        })

    df_g = pd.DataFrame(g_rows)
    df_g.to_csv(OUT_GFEVD, index=False)

    last = gfevd_all[-1]
    gfevd_summary_lines.append("[Existing gfevd_all.npy summary]")
    gfevd_summary_lines.append(f"shape = {gfevd_all.shape}")
    gfevd_summary_lines.append(f"TCI mean = {df_g['TCI'].mean():.6f}")
    gfevd_summary_lines.append(f"TCI std  = {df_g['TCI'].std():.6f}")
    gfevd_summary_lines.append(f"diag_mean mean = {df_g['diag_mean'].mean():.6f}")
    gfevd_summary_lines.append(f"offdiag_mean mean = {df_g['offdiag_mean'].mean():.6f}")
    gfevd_summary_lines.append(f"net_abs_mean avg = {df_g['net_abs_mean'].mean():.6f}")
    gfevd_summary_lines.append(f"row_err max = {df_g['row_err'].max():.12f}")
    gfevd_summary_lines.append("last theta =")
    gfevd_summary_lines.append(np.array2string(last, precision=4, suppress_small=True))
else:
    df_g = None

# =========================
# 4) Summary text
# =========================
lines = []
lines.append("============================================================")
lines.append("TVP-VAR / GFEVD Diagnostic Summary")
lines.append("============================================================")
lines.append(f"beta_t shape : {beta_t.shape}")
lines.append(f"cov_t shape  : {cov_t.shape}")
lines.append(f"lag p        : {p}")
lines.append(f"horizon H    : {H}")
lines.append("")

lines.append("[Covariance summary]")
lines.append(f"diag_mean avg           : {df_cov['diag_mean'].mean():.6f}")
lines.append(f"diag_min global         : {df_cov['diag_min'].min():.6f}")
lines.append(f"eig_min global          : {df_cov['eig_min'].min():.6f}")
lines.append(f"PSD-like ratio          : {df_cov['is_psd_like'].mean():.4f}")
lines.append(f"symmetry max abs diff   : {df_cov['symmetry_max_abs_diff'].max():.12f}")
lines.append("")

lines.append("[Orientation comparison: last MAX_CHECK_T periods]")
lines.append(f"checked periods         : {len(df_orient)}")
lines.append(f"TCI_normal mean         : {df_orient['TCI_normal'].mean():.6f}")
lines.append(f"TCI_transpose mean      : {df_orient['TCI_transpose'].mean():.6f}")
lines.append(f"net_abs_mean_normal     : {df_orient['net_abs_mean_normal'].mean():.6f}")
lines.append(f"net_abs_mean_transpose  : {df_orient['net_abs_mean_transpose'].mean():.6f}")
lines.append(f"diag-offdiag normal     : {df_orient['diag_minus_offdiag_normal'].mean():.6f}")
lines.append(f"diag-offdiag transpose  : {df_orient['diag_minus_offdiag_transpose'].mean():.6f}")
lines.append(f"stable ratio normal     : {df_orient['stable_normal'].mean():.4f}")
lines.append(f"stable ratio transpose  : {df_orient['stable_transpose'].mean():.4f}")
lines.append("")

def verdict_orientation(df):
    # heuristic
    score_normal = 0
    score_tr = 0

    if df["diag_minus_offdiag_normal"].mean() > df["diag_minus_offdiag_transpose"].mean():
        score_normal += 1
    else:
        score_tr += 1

    if df["net_abs_mean_normal"].mean() > df["net_abs_mean_transpose"].mean():
        score_normal += 1
    else:
        score_tr += 1

    if df["stable_normal"].mean() > df["stable_transpose"].mean():
        score_normal += 1
    else:
        score_tr += 1

    if score_normal > score_tr:
        return "normal orientation looks more plausible"
    elif score_tr > score_normal:
        return "transpose orientation looks more plausible"
    else:
        return "inconclusive"

lines.append("[Heuristic orientation verdict]")
lines.append(verdict_orientation(df_orient))
lines.append("")

if GFEVD_FILE.exists():
    lines.extend(gfevd_summary_lines)
    lines.append("")

lines.append("[Interpretation guide]")
lines.append("- eig_min(global) < 0 and PSD-like ratio low: cov_t may not behave like innovation covariance.")
lines.append("- diag_minus_offdiag too small: GFEVD may be overly dense.")
lines.append("- net_abs_mean too small across time: directional asymmetry may be collapsing.")
lines.append("- stable ratio low: TVP-VAR coefficient matrices may be unstable.")
lines.append("- transpose orientation looks better: beta_t block direction may have been interpreted incorrectly.")
lines.append("")

summary_text = "\n".join(lines)
OUT_SUMMARY.write_text(summary_text, encoding="utf-8")

# =========================
# Print
# =========================
print(summary_text)
print(f"\n[INFO] Saved files:")
print(f"  - {OUT_SUMMARY}")
print(f"  - {OUT_ORIENT}")
print(f"  - {OUT_COV}")
print(f"  - {OUT_STAB}")
if GFEVD_FILE.exists():
    print(f"  - {OUT_GFEVD}")

[INFO] beta_t shape = (1035, 7, 7)
[INFO] cov_t  shape = (1035, 7, 7)
[INFO] lag p       = 1


C:\Users\User\AppData\Local\Temp\ipykernel_2620\1865893292.py:145: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  numer += float(term_ij) ** 2
C:\Users\User\AppData\Local\Temp\ipykernel_2620\1865893292.py:148: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  denom += float(term_ii)


TVP-VAR / GFEVD Diagnostic Summary
beta_t shape : (1035, 7, 7)
cov_t shape  : (1035, 7, 7)
lag p        : 1
horizon H    : 10

[Covariance summary]
diag_mean avg           : 1.533146
diag_min global         : 0.073879
eig_min global          : 0.059091
PSD-like ratio          : 1.0000
symmetry max abs diff   : 0.000000000000

[Orientation comparison: last MAX_CHECK_T periods]
checked periods         : 30
TCI_normal mean         : 42.928950
TCI_transpose mean      : 60.187121
net_abs_mean_normal     : 0.200726
net_abs_mean_transpose  : 0.564089
diag-offdiag normal     : 0.499162
diag-offdiag transpose  : 0.297817
stable ratio normal     : 0.9667
stable ratio transpose  : 0.9667

[Heuristic orientation verdict]
transpose orientation looks more plausible

[Existing gfevd_all.npy summary]
shape = (1035, 7, 7)
TCI mean = 52.630168
TCI std  = 7.685382
diag_mean mean = 0.473698
offdiag_mean mean = 0.087717
net_abs_mean avg = 0.149600
row_err max = 0.000000000000
last theta =
[[0.6971 0.1485 0